# Script Loading: `defer`, `async` & Friends

`defer` is a boolean attribute on `<script>` that tells the browser to execute a script only after the HTML document has been fully parsed.

By default, when the parser hits a `<script>` tag it stops building the DOM, downloads the file, runs it, and only then resumes. `defer` removes that stall — the script downloads in parallel while parsing continues.

---

## 1. Key Characteristics of `defer`

- **Non-blocking download.** Fetched in parallel with HTML parsing; the page assembly never pauses.
- **Post-parsing execution.** Runs after the DOM is fully constructed, but strictly *before* `DOMContentLoaded` fires.
- **Preserved order.** Multiple deferred scripts execute sequentially in document order, regardless of which finishes downloading first.
- **External only.** Requires a `src` attribute. On an inline script, `defer` is silently ignored and the script runs immediately.
- **Modules already defer.** `<script type="module">` has deferred behaviour built in, so the attribute is redundant there.

---

## 2. Syntax

Deferred scripts belong in the `<head>` — the browser discovers them early and can start downloading sooner.

```html
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>Defer Example</title>
  <!-- Downloads start immediately; execution waits for the DOM -->
  <script src="config.js" defer></script>
  <script src="main.js" defer></script>
</head>
<body>
  <h1 id="title">Hello World</h1>
</body>
</html>
```

`config.js` is guaranteed to run before `main.js`, and both run after `<h1>` exists.

---

## 3. Normal vs. `async` vs. `defer`

| Script type | HTML parsing | Executes | Order guaranteed |
|---|---|---|---|
| `<script>` | Paused during download **and** execution | Immediately on download | Yes — document order |
| `<script async>` | Paused only during execution | As soon as its download finishes | **No** — first to arrive wins |
| `<script defer>` | Never paused | After parsing completes, before `DOMContentLoaded` | Yes — document order |
| `<script>` before `</body>` | Paused, but the DOM above it already exists | Immediately on download | Yes — document order |

### Timeline

```
defer:   ├── parse HTML ──────────────────┤
         └── download ──┘                 └─ execute ─┤ DOMContentLoaded

async:   ├── parse ─────┤▓▓▓▓├── parse ───┤ DOMContentLoaded
         └── download ──┘  execute
                         (parsing stops wherever the download lands)

normal:  ├── parse ─┤▓▓▓▓▓▓▓▓▓▓▓├── parse ─┤ DOMContentLoaded
                    download+exec
```

### Choosing between them

Use **`defer`** for anything that touches the DOM or depends on another script — which is most application code.

Use **`async`** only for genuinely independent scripts that neither depend on nor are depended upon by anything: analytics tags, error trackers, ad scripts, chat widgets.

Use **neither** when a script must run before the page renders — rare, but real for things like theme-flash prevention or feature-detection polyfills.

The old advice of putting `<script>` at the end of `<body>` achieves a similar result to `defer`, but it's strictly worse: the browser can't discover the file until it has parsed the entire document, so the download starts later.

---

## 4. Why Use `defer`?

**No more null errors.** `document.getElementById('title')` is guaranteed to find the element, because the DOM is complete before the script runs. The classic failure without it:

```js
// <script src="main.js"> in <head>, no defer
const title = document.getElementById('title');
title.textContent = 'Hi';
// TypeError: Cannot set properties of null
```

**Faster rendering.** The structural layout paints without waiting on JavaScript, which improves First Contentful Paint and Largest Contentful Paint.

**No `DOMContentLoaded` wrapper needed.** This is redundant in a deferred script:

```js
document.addEventListener('DOMContentLoaded', () => { ... });  // unnecessary
```

The script already runs at that point in the lifecycle.

---

## 5. Details Worth Knowing

### `defer` waits for the DOM, not for images

`DOMContentLoaded` fires when HTML parsing and deferred scripts are done. Images, stylesheets, iframes, and fonts may still be loading — that's the `load` event. So `img.naturalWidth` can be `0` inside a deferred script.

### Deferred scripts delay `DOMContentLoaded`

The event waits for every deferred script to finish executing. A slow deferred script still holds up anything listening for it, including some analytics timings.

### `document.write` breaks

Calling it from a `defer` or `async` script wipes the document, because the parser has already closed. Don't use it anywhere, but especially not here.

### Modules

```html
<script type="module" src="app.js"></script>              <!-- already deferred -->
<script type="module" src="app.js" async></script>        <!-- opts into async -->
<script type="module">import './app.js';</script>          <!-- inline modules defer too -->
```

Modules are the one case where the deferred behaviour extends to inline scripts. They also run in strict mode automatically and have their own top-level scope, so `var` declarations don't leak to `window`.

The `nomodule` attribute is the legacy pairing — modern browsers ignore a `nomodule` script, old ones ignore `type="module"`:

```html
<script type="module" src="modern.js"></script>
<script nomodule src="legacy.js" defer></script>
```

### Dynamically injected scripts are `async` by default

```js
const s = document.createElement('script');
s.src = 'analytics.js';
s.async = false;   // set explicitly if you need ordered execution
document.head.append(s);
```

### Related resource hints

```html
<link rel="preload"  href="critical.js" as="script">   <!-- fetch early, high priority -->
<link rel="prefetch" href="next-page.js" as="script">  <!-- fetch idle, for a likely next navigation -->
<link rel="modulepreload" href="app.js">               <!-- preload a module and its graph -->
```

These control *fetching* priority; `defer`/`async` control *execution* timing. They solve different problems and are often used together.

---

## 6. Quick Reference

| Need | Use |
|---|---|
| Touches DOM, or scripts depend on each other | `defer` |
| Fully independent third-party tag | `async` |
| Must run before first paint | plain `<script>` in `<head>` |
| ES modules | `type="module"` (deferred already) |
| Ordered dynamic injection | `script.async = false` |

---

## References

- [MDN — `<script>`](https://developer.mozilla.org/en-US/docs/Web/HTML/Element/script)
- [MDN — DOMContentLoaded](https://developer.mozilla.org/en-US/docs/Web/API/Document/DOMContentLoaded_event)
- [MDN — JavaScript modules](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Guide/Modules)
- [javascript.info — Scripts: async, defer](https://javascript.info/script-async-defer)
- [web.dev — Efficiently load third-party JavaScript](https://web.dev/articles/efficiently-load-third-party-javascript)